# Know Your Micro-Watershed

Read a micro-watershed’s area, elevation, terrain, drainage and upstream and downstream connections.

Run the cells in order. Change the place, identifier or columns to explore other records. Downloads from GeoLibre use your selected tehsil; these templates start with Hilsa, Nalanda, Bihar.


## Set up Python

Run the collapsed setup cells. They import the libraries and define `read_json`, a small response reader. It reads JSON text, treats non-standard `NaN` and `Infinity` numbers as missing, and also accepts JSON returned inside a string. HTTP errors and malformed responses remain visible. Expand the cells to read the code.


In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["geopandas", "matplotlib", "requests", "pyodide-http"])
    import pyodide_http
    pyodide_http.patch_all()

import os
import re
import ast
import json
from getpass import getpass
from inspect import isawaitable
from urllib.parse import urljoin
import requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, FileLink
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})


In [ ]:
SCOPE = json.loads("{\"state\": \"Bihar\", \"district\": \"Nalanda\", \"tehsil\": \"Hilsa\"}")
API_URL = 'https://geoserver.core-stack.org/api/v1/'
STAC_URL = 'https://spatio-temporal-asset-catalog.s3.ap-south-1.amazonaws.com/CorestackCatalogs_merged_collection/tehsil_wise/catalog.json'
YEARS = list(range(2017, 2025))


In [ ]:
"""Small response reader embedded in the notebooks' collapsed setup cell."""
import json


def read_json(response):
    """Read JSON text; represent non-standard NaN/Infinity values as missing."""
    response.raise_for_status()
    raw_text = response.text.lstrip("\ufeff")
    try:
        # Some API tables contain bare NaN or Infinity, which are not JSON numbers.
        data = json.loads(raw_text, parse_constant=lambda value: None)
        # Also accept a JSON document returned as a JSON-encoded string.
        if isinstance(data, str):
            data = json.loads(data.lstrip("\ufeff"), parse_constant=lambda value: None)
        return data
    except ValueError as error:
        raise ValueError(
            "The server response is not readable JSON. "
            "Inspect response.status_code and response.text[:500], then retry the request."
        ) from error


## Choose the place and set your API key

Edit `SCOPE` in the setup cell to change the place. The [public API guide](https://docs.core-stack.org/use-precomputed-data/public-apis/) explains registration and API keys. This cell reuses `CORE_STACK_API_KEY` or asks for it privately, then stores it in this kernel’s environment. The key is sent only to the API, in the `X-API-Key` header. Restart the kernel and run from the top after changing places.


In [ ]:
place = {key: re.sub(r"[\s_]+", "_", SCOPE[key].replace("(", "").replace(")", "")).strip("_").lower()
         for key in ["state", "district", "tehsil"]}
state, district, tehsil = place["state"], place["district"], place["tehsil"]
api_key = os.environ.get("CORE_STACK_API_KEY", "").strip()
if not api_key:
    api_key = getpass("CoRE Stack API key: ")
    if isawaitable(api_key):
        api_key = await api_key
os.environ["CORE_STACK_API_KEY"] = str(api_key).strip()
api_headers = {"X-API-Key": os.environ["CORE_STACK_API_KEY"]}
display(place)


## Read the tehsil and choose a micro-watershed

One request returns the tehsil’s tables. The cell keeps the tables used here, lists identifiers and shows the first record’s first ten fields. Change the selected identifier, then rerun the following cells. Blank fields mean the source did not supply a value.


In [ ]:
response = requests.get(API_URL + "get_tehsil_data/", params=place, headers=api_headers, timeout=180)
api_data = read_json(response)
required_tables = ['mws', 'dem', 'terrain', 'mws_connectivity', 'drainage_density', 'stream_order', 'river', 'canal', 'mws_intersect_villages', 'mws_intersect_swb']
tables = {name: pd.DataFrame(api_data.get(name, [])) for name in required_tables}
display(pd.DataFrame({"Table": required_tables, "Rows": [len(tables[name]) for name in required_tables]}))
mws_table = tables['mws']
display(mws_table[["uid"]])
mws_id = str(mws_table.iloc[0]["uid"])  # Choose another ID from the list.
selected = mws_table.loc[mws_table["uid"].astype(str) == mws_id].iloc[0]
display(selected.iloc[:10].to_frame("First 10 fields"))


## Discover data and descriptions in STAC

STAC lists published datasets, field descriptions, downloads and styles. Change `dataset` to another item from the collection. Asset links are used as published, wherever the files are hosted. STAC describes asset fields; API tables may use different names and units, which are shown explicitly in the examples below.


In [ ]:
collection_url = urljoin(STAC_URL, f"{state}/{district}/{tehsil}/collection.json")
response = requests.get(collection_url, timeout=90)
collection = read_json(response)
items = pd.DataFrame([{"Item": link["href"].split("/")[-1].removesuffix(".json"),
                       "URL": urljoin(collection_url, link["href"])}
                      for link in collection["links"] if link["rel"] == "item"], columns=["Item", "URL"])
# Follow a relevant item link from the collection.
dataset = "terrain_vector"
matches = items.loc[items["Item"].str.endswith("_" + dataset)]
item = None
if not matches.empty:
    item_url = matches.iloc[0]["URL"]
    response = requests.get(item_url, timeout=90)
    item = read_json(response)
    display(Markdown(item["properties"].get("description", "No description published.")))
    field_notes = pd.DataFrame(item["properties"].get("table:columns", []))
    display(field_notes.reindex(columns=["name", "type", "description"]).head(12))
    print("Published field count:", len(field_notes), "— use field_notes to see them all.")
    display(pd.DataFrame(item["assets"]).T.reindex(columns=["title", "type", "href"]))
else:
    print("This dataset is not listed in the tehsil's STAC collection. Available items:")
    display(items)


## Area, basin and elevation

Areas are hectares and elevations are metres. Basin codes describe the source’s basin hierarchy.


In [ ]:
display(selected.reindex(["area_in_ha", "watershed_code", "basin_code", "sub_basin_code"]).to_frame("Value"))
elevation = tables["dem"].set_index("uid").reindex([mws_id])
display(elevation.reindex(columns=["min_elevation_in_m", "mean_elevation_in_m", "max_elevation_in_m"]))


## How much is plain, slope, ridge or valley?

These API fields give each terrain class as a percentage of MWS area. The doughnut shows a complete set of shares totalling about 100%. Otherwise, a bar chart shows the available values without rescaling them.


In [ ]:
terrain = tables["terrain"].set_index("uid").reindex([mws_id]).iloc[0]
terrain_fields = {"plain_area_percent": "Plains", "slopy_area_percent": "Broad slopes",
                  "hill_slope_area_percent": "Hill slopes", "ridge_area_percent": "Ridges", "valley_area_percent": "Valleys"}
terrain_shares = pd.to_numeric(terrain.reindex(terrain_fields), errors="coerce").rename(index=terrain_fields)
display(terrain_shares.to_frame("MWS area (%)"))
print("Reported total (%):", terrain_shares.sum(min_count=1))
if terrain_shares.notna().all() and (terrain_shares >= 0).all() and abs(terrain_shares.sum() - 100) < 0.5:
    terrain_shares.dropna().plot.pie(figsize=(6, 5), autopct="%1.1f%%", ylabel="", startangle=90,
                                   wedgeprops={"width": 0.45}, title=f"Terrain · {mws_id}")
    plt.show()
else:
    terrain_shares.dropna().plot.barh(figsize=(7, 3), xlabel="Reported MWS area (%)", title="Available terrain shares")
    plt.tight_layout()
    plt.show()


## Where does the water flow?

Select the published upstream and downstream identifiers, then highlight them on the boundary map. A missing connectivity record does not mean that no connections exist.


In [ ]:
connections = tables["mws_connectivity"]
connection = connections.loc[connections["uid"].astype(str) == mws_id]
upstream_ids, downstream_ids = [], []
if not connection.empty:
    record = connection.iloc[0]
    upstream_ids = ast.literal_eval(record["upstream_mws"]) if pd.notna(record["upstream_mws"]) else []
    downstream_ids = [record["downstream_mws"]] if pd.notna(record["downstream_mws"]) and record["downstream_mws"] else []
    display(pd.Series({"Upstream MWS": upstream_ids, "Downstream MWS": downstream_ids}).to_frame("Identifiers"))
else:
    print("No connectivity record was returned for this MWS.")
response = requests.get(API_URL + "get_mws_geometries/", params=place, headers=api_headers, timeout=180)
boundaries = gpd.GeoDataFrame.from_features(read_json(response)["features"], crs="EPSG:4326")
boundaries["uid"] = boundaries["uid"].astype(str)
related = boundaries.loc[boundaries["uid"].isin(upstream_ids + downstream_ids + [mws_id])].copy()
related["Connection"] = "Selected MWS"
related.loc[related["uid"].isin(upstream_ids), "Connection"] = "Upstream"
related.loc[related["uid"].isin(downstream_ids), "Connection"] = "Downstream"
if not related.empty:
    related.plot(column="Connection", categorical=True, legend=True, edgecolor="white", figsize=(7, 6))
    plt.title(f"Water connections · {mws_id}")
    plt.axis("off")
    plt.show()
print("Connected IDs outside these returned boundaries:", sorted(set(upstream_ids + downstream_ids) - set(boundaries["uid"])))


## Drainage and stream orders

Compare the recorded drainage-density measures and stream-order shares. Stream-order fields are labelled as area percentages in the API; they are not percentages of stream length.


In [ ]:
drainage = tables["drainage_density"].set_index("uid").reindex([mws_id]).iloc[0]
streams = tables["stream_order"].set_index("uid").reindex([mws_id]).iloc[0]
display(drainage.reindex(["drainage_density_weighted_in_km_per_km2", "drainage_density_std_in_km_per_km2", "stream_order_length_in_km"]).to_frame("Value"))
orders = [f"order_{n}_area_percent" for n in range(1, 12)]
shares = pd.to_numeric(streams.reindex(orders), errors="coerce")
shares.index = range(1, 12)
shares.dropna().plot.bar(figsize=(8, 3), color="#287d8e", xlabel="Stream order", ylabel="Area (%)", title=f"Stream orders · {mws_id}")
plt.tight_layout()
plt.show()


## Inspect rivers, canals and linked records

Change `table_name` to `canal`, `mws_intersect_villages` or `mws_intersect_swb`. These tables retain their published field names; the village intersection table uses `mws uid`.


In [ ]:
table_name = "river"
table = tables[table_name]
id_column = "mws uid" if table_name == "mws_intersect_villages" else "uid"
display(table.loc[table[id_column].astype(str) == mws_id].T)


## Get a compact indicator record

The KYL indicator API offers another view of this MWS. Its own field names remain visible.


In [ ]:
response = requests.get(API_URL + "get_mws_kyl_indicators/", params={**place, "mws_id": mws_id}, headers=api_headers, timeout=180)
indicators = pd.json_normalize(read_json(response))
display(indicators.T)


## Look up the MWS from a point

Use a point inside the selected boundary, or replace the coordinates with your own latitude and longitude.


In [ ]:
selected_boundary = boundaries.loc[boundaries["uid"] == mws_id]
if not selected_boundary.empty:
    point = selected_boundary.geometry.iloc[0].representative_point()
    coordinates = {"latitude": point.y, "longitude": point.x}
    response = requests.get(API_URL + "get_mwsid_by_latlon/", params=coordinates, headers=api_headers, timeout=90)
    display(read_json(response) if response.ok else {"HTTP status": response.status_code, "Response": response.text[:500]})


## Open an available MWS report

This API returns a report link when one is available. A missing report does not prevent the data examples above from running.


In [ ]:
response = requests.get(API_URL + "get_mws_report/", params={**place, "mws_id": mws_id}, headers=api_headers, timeout=90)
display(read_json(response) if response.ok else {"HTTP status": response.status_code, "Response": response.text[:500]})
